# Car Detection

This notebook demonstrates how to detect cars in satellite imagery using the geoai package.

In [ ]:
# import the geoai package
import geoai

# download the raster dataset

raster_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/cars_7cm.tif"
)

raster_path = geoai.download_file(raster_url)

In [ ]:
# Move the downloaded files to the target folder

import os
import shutil
from pathlib import Path

# ==== INPUTS ====

target_folder = "data/geoai/car-detection"  # Enter your target folder path here
datasets = ["cars_7cm.tif"]  # Add dataset names as a list
source_dir = Path(".")  # Current location of datasets

# ==== SCRIPT ====

target_path = Path(target_folder)
target_path.mkdir(parents=True, exist_ok=True)
for dataset_name in datasets:
    source_path = source_dir / dataset_name
    
    if not source_path.exists():
        print(f"Dataset '{dataset_name}' not found in {source_dir}")
        continue
    
    if source_path.is_file():
        files_to_move = [source_path]
    else:
        files_to_move = list(source_path.parent.glob(f"{dataset_name}.*"))
    
    for file_path in files_to_move:
        dest = target_path / file_path.name
        shutil.move(str(file_path), str(dest))
        print(f"Moved: {file_path.name} -> {target_folder}")
        
print("Done!")

raster = ("data/geoai/car-detection/cars_7cm.tif")

## Visualize the image

In [ ]:
geoai.view_raster(raster)

In [ ]:
# Initialize the model

detector = geoai.CarDetector()

## Extract cars

Extract cars from the image using the model and save the output image.

In [ ]:
mask_path = detector.generate_masks(
    raster_path=raster,
    output_path="data/geoai/car-detection/cars_masks.tif",
    confidence_threshold=0.3,
    mask_threshold=0.5,
    overlap=0.25,
    chip_size=(400, 400),
)

Convert the image masks to polygons and save the output GeoJSON file.

In [ ]:
gdf = detector.vectorize_masks(
    masks_path="data/geoai/car-detection/cars_masks.tif",
    output_path="data/geoai/car-detection/cars.geojson",
    min_object_area=100,
    max_object_area=2000,
    # n_workers=12, # set n_workers to run in parallel
)

In [ ]:
# Add geometric properties

gdf = geoai.add_geometric_properties(gdf)

In [ ]:
#Visualize initial results

geoai.view_vector_interactive(gdf, column="confidence", tiles=raster_url)

In [ ]:
# Filter cars by area

gdf_filter = gdf[
    (gdf["area_m2"] > 8) & (gdf["area_m2"] < 60) & (gdf["minor_length_m"] > 1)
]
len(gdf_filter)

## Visualize final results

In [ ]:
geoai.view_vector_interactive(gdf_filter, column="confidence", tiles=raster_url)

In [ ]:
geoai.view_vector_interactive(gdf_filter, tiles=raster_url)